In [1]:
import os

os.environ["OMP_NUM_THREADS"] = "6"
os.environ["MKL_NUM_THREADS"] = "6"
os.environ["OPENBLAS_NUM_THREADS"] = "6"
os.environ["NUMEXPR_NUM_THREADS"] = "6"

import torch
torch.set_num_threads(6)
torch.set_num_interop_threads(1)

In [2]:
import sys

import torch

import pandas as pd

import pm4py

from config.feature_config import FeatureConfig
from config.ga_config import GAConfig

from utils.general_utils import set_stdout_to_file, set_seed
from utils.feature_utils import df_to_sequence_array

from model.preprocessor import PreprocessorArtifacts

from model.next_event_model import ProcessLSTM
from model.model_wrapper import ModelWrapper

from ga_search.search import CounterfactualGA

from process.engine import ProcessModelConstraintEngine
from process.experimenter import ExperimentHandler

### --- Load Dataset & Models ---

In [3]:
set_seed(seed=777)

In [4]:
df = pd.read_excel(
    "../../../data/sepsis.xlsx",
    engine="openpyxl",
    keep_default_na=False,
    dtype={
        "case:concept:name": "string",
        "concept:name": "string",
        "lifecycle:transition": "string",
        "org:group": "string",
        "case:age": "float32",
        "Leucocytes": "float32",
        "CRP": "float32",
    }
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [5]:
df.head(20)

,case:concept:name,time:timestamp,CRP,Leucocytes,case:age,concept:name,lifecycle:transition,org:group,time_delta
0,A,2014-10-22 11:15:41,0.0,0.0,85.0,ER Registration,complete,A,0
1,A,2014-10-22 11:27:00,0.0,9.6,85.0,Leucocytes,complete,B,679
2,A,2014-10-22 11:27:00,21.0,0.0,85.0,CRP,complete,B,0
3,A,2014-10-22 11:27:00,0.0,0.0,85.0,LacticAcid,complete,B,0
4,A,2014-10-22 11:33:37,0.0,0.0,85.0,ER Triage,complete,C,397
5,A,2014-10-22 11:34:00,0.0,0.0,85.0,ER Sepsis Triage,complete,A,23
6,A,2014-10-22 14:03:47,0.0,0.0,85.0,IV Liquid,complete,A,8987
7,A,2014-10-22 14:03:47,0.0,0.0,85.0,IV Antibiotics,complete,A,0
8,A,2014-10-22 14:13:19,0.0,0.0,85.0,Admission NC,complete,D,572
9,A,2014-10-24 09:00:00,109.0,0.0,85.0,CRP,complete,B,154001


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [7]:
feature_config = FeatureConfig.load(
    path = "../pretrained_models/"
)

In [8]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['CRP', 'Leucocytes', 'case:age', 'concept:name', 'lifecycle:transition', 'org:group', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
org:group                      categorical    event    yes    ['A', 'B', 'C', ...]                     N/A        data_derived        
case:age                       continuous     case     yes    [40.00, 90.00]                           10.0000    quantile_derived    
time_delta                     continuous     event    yes    [0.00, 38748.80]                         139.0000   quantile_derived    
Leucocytes                    

In [9]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../pretrained_models/"
)

In [10]:
model = ProcessLSTM.load(
    path = "../pretrained_models/"
)

In [11]:
model_wrapper = ModelWrapper(
    model=model,
    preprocessor_artifacts=preprocessor_artifacts,
    device=device
)

### --- Process Constraints ---

In [12]:
engine = ProcessModelConstraintEngine.load(
    path = "../pretrained_models/"
)

In [13]:
engine.parallel_sets

[{'Admission NC',
  'CRP',
  'ER Registration',
  'ER Sepsis Triage',
  'ER Triage',
  'IV Antibiotics',
  'IV Liquid',
  'LacticAcid',
  'Leucocytes'},
 {'ER Registration', 'ER Sepsis Triage', 'ER Triage', 'IV Antibiotics'}]

In [14]:
engine.branching_sets

[{'Admission NC',
  'CRP',
  'ER Registration',
  'ER Sepsis Triage',
  'ER Triage',
  'IV Antibiotics',
  'IV Liquid',
  'LacticAcid',
  'Leucocytes'},
 {'ER Registration', 'ER Sepsis Triage', 'ER Triage', 'IV Antibiotics'},
 {'Release C', 'Release D', 'Release E'}]

### --- Experiments Generation ---

In [15]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/sepsis-cf_seed777_experiments_ga_ablated_output.txt", console=False)

In [16]:
generator = ExperimentHandler(
    constraint_engine=engine,
)

In [17]:
exp_df_sin, metadata_sin = ExperimentHandler.load("../experiments/cf_generated_experiments_single_desired")
print("Mined using parameters:", metadata_sin["parameters"])

### --- Counterfactuals ---

In [18]:
ga_config = GAConfig(
    w_distance=1.0,
    w_sparsity=1.0,
    w_margin=1.0,
    w_process_violation=0.0,
)
ga_config.validate()

cf_GA = CounterfactualGA(
    ga_config=ga_config,
    feature_config=feature_config,
    model_wrapper=model_wrapper
)

In [19]:
results_sin = generator.run_experiment_df(
    cf_method=cf_GA,
    technique="GA_Ablated_single_desired_seed777",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/190 [00:00<?, ?case/s]

In [20]:
results_sin

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,FFA,3,1,3,0.543182,0.186364,0.900000,0.388889,0.583333,...,0.361111,0.000000,0.250000,0.500000,0.000000,0.111111,0.0,0.000000,0.0,0.000000
1,0,CL,4,1,4,0.467774,0.135548,0.800000,0.322222,0.213636,...,0.361111,0.181818,0.250000,0.500000,0.000000,0.111111,0.0,0.888432,0.0,1.000000
2,0,KY,5,1,5,0.469929,0.139859,0.800000,0.372222,0.307692,...,0.361111,0.307692,0.250000,0.500000,0.000000,0.111111,0.0,0.867850,0.0,1.000000
3,0,FP,6,1,6,0.468578,0.112157,0.825000,0.405556,0.400000,...,0.476933,0.400000,0.254711,0.500000,0.009422,0.222222,0.0,0.000000,0.0,0.000000
4,0,GIA,7,1,7,0.522920,0.145840,0.900000,0.400000,0.350000,...,0.361111,0.352941,0.250000,0.500000,0.000000,0.111111,0.0,0.901160,0.0,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
162,18,MFA,12,1,11,0.273284,0.096569,0.450000,0.335556,0.600000,...,0.123808,0.407407,0.034919,0.000000,0.069838,0.088889,0.0,0.919620,0.0,1.000000
163,18,OKA,12,1,11,0.266417,0.091925,0.440909,0.312222,0.507407,...,0.160913,0.407407,0.072024,0.090909,0.053139,0.088889,0.0,0.766029,0.0,0.999999
164,18,SS,12,1,11,0.281511,0.108477,0.454545,0.300000,0.475926,...,0.202811,0.407407,0.091699,0.090909,0.092490,0.111111,0.0,0.905628,0.0,1.000000
165,18,ZFA,12,1,11,0.268759,0.110245,0.427273,0.312222,0.581481,...,0.207157,0.703704,0.096046,0.090909,0.101182,0.111111,0.0,0.000000,0.0,0.000000


### --- Cleanup ---

In [21]:
sys.stdout = original_stdout
log_file.close()